<div style="text-align: center;">
    <img src="img/giphy.gif" width="300">
    <p>Image from: <a href="https://media0.giphy.com/media/khKHUbMYTXLb2/giphy.gif?cid=ecf05e473wejqtna3hppengb1mihpcgvh4sack7rjro597ho&rid=giphy.gif&ct=g">source</a></p>
</div>

# Exploratory Data Analysis of the Disney Dataset

## Foreword

This notebook demonstrates the process of exploratory data analysis (EDA) using the [Disney Movies Dataset](https://www.kaggle.com/datasets/suvroo/disney-movies-dataset). It provides an example of how to organize and present findings in a project report for this course. Through data cleaning, analysis, and visualization, this notebook explores the dataset to uncover interesting patterns and trends.

# Introduction

## Questions of Interest

In this analysis, I will explore the Disney dataset to better understand the performance of Disney movies and trends in Disney's entertainment business. I am particularly interested in identifying which movie genres and directors have achieved the greatest commercial success. Before analyzing the data, I believe The Lion King may be one of Disney's best-selling movies, given its popularity and cultural impact. I also expect action and adventure movies to generate the highest box office earnings. Regarding directors, I predict that John Lasseter may rank among the top performers because of his reputation as a well-known and successful Disney director.

These questions are interesting because Disney has produced movies across a wide range of genres and expanded its business into several entertainment-related areas. By examining the data, I hope to identify patterns in movie performance, compare directors based on their movies' box office earnings and genres, and understand how Disney's revenue contributions across different business segments have changed over time.

## Dataset description 

The datasets used in this project were provided by the course.

The Disney dataset is composed of $5$ tables, `disney_movies_total_gross.csv`, `disney_revenue_1991-2016.csv`, `disney-characters.csv`, `disney-director.csv`, `disney-voice-actors.csv`. Each table is stored in a `.csv` file and contains different information about Disney pieces including movies, revenues, directors, characters, voice-actors. I will be using the `movies` , `characters` and `directors` tables formally described below:

* **disney_movies_total_gross.csv**
    * Contains information about Disney movies, including movie titles, release dates, genres, MPAA ratings, total box office gross, and inflation-adjusted gross.
* **disney-director.csv**
    * Contains information about Disney movies and their directors, allowing comparisons of directors' movie counts and box office performance.
* **disney-characters.csv**
    * Contains information about Disney movie characters and their associated films, providing an additional perspective on the characters featured in Disney's movie collection.
 

In [1]:
# import the required libraries
import altair as alt
import pandas as pd
import numpy as np
from clean_number import clean_number_columns
from clean_text import clean_text_columns

In [2]:
# import csv files
movies = pd.read_csv('data/disney_movies_total_gross.csv')
directors=pd.read_csv('data/disney-director.csv')
characters=pd.read_csv('data/disney-characters.csv')

<br>

# Examine the structure and information of the data

<br>

In [3]:
movies.shape

(579, 6)

In [4]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 579 entries, 0 to 578
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   movie_title               579 non-null    object
 1   release_date              579 non-null    object
 2   genre                     562 non-null    object
 3   MPAA_rating               523 non-null    object
 4   total_gross               579 non-null    object
 5   inflation_adjusted_gross  579 non-null    object
dtypes: object(6)
memory usage: 27.3+ KB


In [5]:
movies.describe()

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross
count,579,579,562,523,579,579
unique,573,553,12,5,576,576
top,The Jungle Book,"Dec 25, 1997",Comedy,PG,$0,$0
freq,3,3,182,187,4,4


In [6]:
# check blank values
movies[movies.isna().any(axis=1)]

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross
5,"20,000 Leagues Under the Sea","Dec 23, 1954",Adventure,NaN,"$28,200,000","$528,279,994"
7,Sleeping Beauty,"Jan 29, 1959",Drama,NaN,"$9,464,608","$21,505,832"
9,The Absent Minded Professor,"Mar 16, 1961",Comedy,NaN,"$25,381,407","$310,094,574"
12,The Sword in the Stone,"Dec 25, 1963",Adventure,NaN,"$22,182,353","$153,870,834"
14,Blackbeard's Ghost,"Feb 8, 1968",Comedy,NaN,"$21,540,050","$138,612,686"
...,...,...,...,...,...,...
185,It's Pat,"Aug 26, 1994",Comedy,NaN,"$60,822","$125,666"
251,The War at Home,"Nov 20, 1996",NaN,R,"$34,368","$65,543"
304,Endurance,"May 14, 1999",NaN,PG,"$229,128","$380,218"
350,High Heels and Low Lifes,"Oct 26, 2001",NaN,R,"$226,792","$337,782"


In [7]:
# check if null value exists
movies.isnull().sum()

movie_title                  0
release_date                 0
genre                       17
MPAA_rating                 56
total_gross                  0
inflation_adjusted_gross     0
dtype: int64

In [8]:
# check duplicate movies
movies[movies['movie_title'].duplicated(keep=False)].sort_values(
    by='movie_title'
)


,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross
8,101 Dalmatians,"Jan 25, 1961",Comedy,G,"$153,000,000","$1,362,870,985"
252,101 Dalmatians,"Nov 27, 1996",Comedy,G,"$136,189,294","$258,728,898"
196,Bad Company,"Jan 20, 1995",Thriller/Suspense,R,"$3,523,909","$6,829,092"
364,Bad Company,"Jun 7, 2002",Action,PG-13,"$30,157,016","$43,756,218"
4,Cinderella,"Feb 15, 1950",Drama,G,"$85,000,000","$920,608,730"
556,Cinderella,"Mar 13, 2015",Drama,PG,"$201,151,353","$201,151,353"
19,Freaky Friday,"Jan 21, 1977",Comedy,NaN,"$25,942,000","$98,067,733"
386,Freaky Friday,"Aug 6, 2003",Comedy,PG,"$110,222,438","$154,090,360"
13,The Jungle Book,"Oct 18, 1967",Musical,Not Rated,"$141,843,000","$789,612,346"
194,The Jungle Book,"Dec 25, 1994",Adventure,PG,"$44,342,956","$88,930,321"


<br>
  
After examining the movie table, I found that 66 rows contain missing values and 11 movie titles are duplicated. Therefore, 
we need to clean the movie dataset by addressing the missing values before merging it with the other tables. I will keep duplicated movies as they are released in different year.

<br> 

## 1. Clean the Movies Dataset and Handle Missing Values

<br>

In [9]:
# fill blank with proper values in genre and MPAA_rating columns
movies['genre']=movies['genre'].fillna('No Genre').astype(str)
movies['MPAA_rating']=movies['MPAA_rating'].fillna('Not Rated').astype(str)

In [10]:
# check if null value exists
movies.isnull().sum()

movie_title                 0
release_date                0
genre                       0
MPAA_rating                 0
total_gross                 0
inflation_adjusted_gross    0
dtype: int64

<br>

Convert date column in movies,and split it by year, month, day  
  
<br>

In [11]:
# split release_date into year, month and day
movies['release_date'] = pd.to_datetime(
    movies['release_date'],
    format='%b %d, %Y'
)
movies['year'] = movies['release_date'].dt.year
movies['month'] = movies['release_date'].dt.month_name()
movies['day'] = movies['release_date'].dt.day

# clean number columns
movies=clean_number_columns(movies,['total_gross','inflation_adjusted_gross'])

In [12]:
# create year range
movies.loc[movies['year'] < 1950, 'year_range'] = 'Before 1950'
movies.loc[(movies['year'] >= 1950) & (movies['year'] < 1960), 'year_range'] = '1950-1959'
movies.loc[(movies['year'] >= 1960) & (movies['year'] < 1970), 'year_range'] = '1960-1969'
movies.loc[(movies['year'] >= 1970) & (movies['year'] < 1980), 'year_range'] = '1970-1979'
movies.loc[(movies['year'] >= 1980) & (movies['year'] < 1990), 'year_range'] = '1980-1989'
movies.loc[(movies['year'] >= 1990) & (movies['year'] < 2000), 'year_range'] = '1990-1999'
movies.loc[(movies['year'] >= 2000) & (movies['year'] < 2010), 'year_range'] = '2000-2009'
movies.loc[(movies['year'] >= 2010) & (movies['year'] <= 2016), 'year_range'] = '2010-2016'
movies.head()

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross,year,month,day,year_range
0,Snow White and the Seven Dwarfs,1937-12-21,Musical,G,184925485.0,5.228953e+09,1937,December,21,Before 1950
1,Pinocchio,1940-02-09,Adventure,G,84300000.0,2.188229e+09,1940,February,9,Before 1950
2,Fantasia,1940-11-13,Musical,G,83320000.0,2.187091e+09,1940,November,13,Before 1950
3,Song of the South,1946-11-12,Adventure,G,65000000.0,1.078511e+09,1946,November,12,Before 1950
4,Cinderella,1950-02-15,Drama,G,85000000.0,9.206087e+08,1950,February,15,1950-1959


In [13]:
# check output
movies['year_range'].value_counts().sort_index()

year_range
1950-1959        4
1960-1969        7
1970-1979        9
1980-1989       61
1990-1999      236
2000-2009      172
2010-2016       86
Before 1950      4
Name: count, dtype: int64

<br>

## 2. Clean and Prepare the directors Dataset

<br>

In [14]:
directors.shape

(56, 2)

In [15]:
directors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      56 non-null     object
 1   director  56 non-null     object
dtypes: object(2)
memory usage: 1.0+ KB


In [16]:
directors.describe()

,name,director
count,56,56
unique,56,29
top,Snow White and the Seven Dwarfs,Wolfgang Reitherman
freq,1,7


In [17]:
directors=clean_text_columns(directors,['name','director'])

In [18]:
directors.head()

,name,director
0,Snow White and the Seven Dwarfs,David Hand
1,Pinocchio,Ben Sharpsteen
2,Fantasia,full credits
3,Dumbo,Ben Sharpsteen
4,Bambi,David Hand


<br>

## 3. Clean and Prepare the characters Dataset

<br>

In [19]:
characters.shape

(56, 5)

In [20]:
characters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   movie_title   56 non-null     object
 1   release_date  56 non-null     object
 2   hero          52 non-null     object
 3   villian       46 non-null     object
 4   song          47 non-null     object
dtypes: object(5)
memory usage: 2.3+ KB


In [21]:
characters.describe()

,movie_title,release_date,hero,villian,song
count,56,56,52,46,47
unique,56,56,49,46,47
top,\nSnow White and the Seven Dwarfs,"December 21, 1937",Donald Duck,Evil Queen,Some Day My Prince Will Come
freq,1,1,2,1,1


In [22]:
characters.head()

,movie_title,release_date,hero,villian,song
0,\nSnow White and the Seven Dwarfs,"December 21, 1937",Snow White,Evil Queen,Some Day My Prince Will Come
1,\nPinocchio,"February 7, 1940",Pinocchio,Stromboli,When You Wish upon a Star
2,\nFantasia,"November 13, 1940",NaN,Chernabog,NaN
3,Dumbo,"October 23, 1941",Dumbo,Ringmaster,Baby Mine
4,\nBambi,"August 13, 1942",Bambi,Hunter,Love Is a Song


In [23]:
characters=clean_text_columns(characters,['movie_title','hero','villian','song'])

In [24]:
characters.head()

,movie_title,release_date,hero,villian,song
0,Snow White and the Seven Dwarfs,"December 21, 1937",Snow White,Evil Queen,Some Day My Prince Will Come
1,Pinocchio,"February 7, 1940",Pinocchio,Stromboli,When You Wish upon a Star
2,Fantasia,"November 13, 1940",NaN,Chernabog,NaN
3,Dumbo,"October 23, 1941",Dumbo,Ringmaster,Baby Mine
4,Bambi,"August 13, 1942",Bambi,Hunter,Love Is a Song


<br>

## 4. Join movies, directors and characters dataset

<br>

In [25]:
# movies join directors
movies_dir= pd.merge(
    movies,
    directors,
    left_on='movie_title',
    right_on='name',
    how='left'
).drop(columns=['name'])


In [26]:
# join with characters
movies_dir_char = pd.merge(
    movies_dir,
    characters[['movie_title','hero', 'villian','song']],
    on='movie_title',
    how='left'
)


In [27]:
movies_dir_char.shape

(579, 14)

In [28]:
movies_dir_char.head()

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross,year,month,day,year_range,director,hero,villian,song
0,Snow White and the Seven Dwarfs,1937-12-21,Musical,G,184925485.0,5.228953e+09,1937,December,21,Before 1950,David Hand,Snow White,Evil Queen,Some Day My Prince Will Come
1,Pinocchio,1940-02-09,Adventure,G,84300000.0,2.188229e+09,1940,February,9,Before 1950,Ben Sharpsteen,Pinocchio,Stromboli,When You Wish upon a Star
2,Fantasia,1940-11-13,Musical,G,83320000.0,2.187091e+09,1940,November,13,Before 1950,full credits,NaN,Chernabog,NaN
3,Song of the South,1946-11-12,Adventure,G,65000000.0,1.078511e+09,1946,November,12,Before 1950,NaN,NaN,NaN,NaN
4,Cinderella,1950-02-15,Drama,G,85000000.0,9.206087e+08,1950,February,15,1950-1959,Wilfred Jackson,Cinderella,Lady Tremaine,Bibbidi-Bobbidi-Boo


# Analysis and Discussion of Results

## 1. Movie Genre Analysis

In [29]:
# make a list of year range order to reuse
year_range_order = [
    'Before 1950',
    '1950-1959',
    '1960-1969',
    '1970-1979',
    '1980-1989',
    '1990-1999',
    '2000-2009',
    '2010-2016'
]

In [30]:
chart = alt.Chart(movies_dir_char).mark_bar().encode(
    x=alt.X(
        'year_range:O',
        sort=year_range_order,
        title='Year Range'
    ),
    y=alt.Y('inflation_adjusted_gross:Q', title='Total Inflation Adjusted Gross'),
    color=alt.Color('genre:N', title='Genre'),
    tooltip=['year_range', 'genre', 'inflation_adjusted_gross']
).properties(
    title='Gross by Year Range and Genre',
    width=500,
    height=300
)


In [31]:
chart

alt.Chart(...)

<br>

### Key Findings: Disney Movie Genres Trends and Inflation Adjusted Gross

1. The distribution of Disney movie genres has changed over the years, reflecting how audience preferences and popular genres have evolved over time.

2. Before 1950, adventure and musical movies were popular. Drama became more prominent during the 1950s, while comedy gained popularity in the 1960s and maintained a relatively stable share in the following decades. From 2010 to 2016, adventure and action movies became increasingly popular, while drama continued to play an important role.

3. Overall, adventure, action, and drama were among the major genres contributing to Disney's box office gross over time. These findings suggest that Disney has adapted its movie production to changing audience preferences while continuing to achieve commercial success across different genres.

<br>

## 2. Movie inflation_adjusted_gross Analysis 

### 2.1 Compare the inflation adjusted gross with total grass

In [33]:
# group movies_dir_char dataset by year range
gross_by_range = (
    movies_dir_char.groupby('year_range', observed=False)[
        ['total_gross', 'inflation_adjusted_gross']
    ]
    .sum()
    .reset_index()
)

# chart 1 shows total gross in year range
chart1 = alt.Chart(gross_by_range).mark_area(color='#AAD692').encode(
    x=alt.X('year_range:N', sort=year_range_order, title='Year Range'),
    y=alt.Y('total_gross:Q', title='Total Gross'),
    tooltip=['year_range', 'total_gross']
).properties(
    title='Total Gross by Year Range',
    width=400,
    height=300
)

# chart 3 shows inflation adjusted gross in year range
chart2 = alt.Chart(gross_by_range).mark_area(color='#ADD8E6').encode(
    x=alt.X('year_range:N', sort=year_range_order, title='Year Range'),
    y=alt.Y('inflation_adjusted_gross:Q', title='Inflation-Adjusted Gross'),
    tooltip=['year_range', 'inflation_adjusted_gross']
).properties(
    title='Inflation-Adjusted Gross by Year Range',
    width=400,
    height=300
)

# visualize both charts
chart1 | chart2

alt.HConcatChart(...)

<br>

### Key Findings: Movies Total Gross vs. Inflation Adjusted Gross

1. The inflation-adjusted gross shows a different picture of Disney movie earnings over the years compared with the original total gross.

2. Compared with the original total gross, the inflation-adjusted figures show a dramatic increase in movie earnings before 1950, followed by a decline after 1999.

3. The lowest point occurred during 1970–1979, while the peak period was 1990–1999.

4. The comparison suggests that Disney movies released between 1937 and 1950 performed much better in inflation-adjusted terms than the original gross figures might suggest.

<br>

## 3. Director Analysis with inflation adjusted gross

### 3.1 Directors Ranked by Inflation Adjusted Box Office Gross

In [34]:
# group movies_dir_char by director and sum up inflation adjusted gross
director_stats_top10 = (
    movies_dir_char.groupby('director')
    .agg(
        movie_count=('movie_title', 'count'),
        inflation_adjusted_gross=('inflation_adjusted_gross', 'sum')
    )
    .reset_index()
    .sort_values('inflation_adjusted_gross', ascending=False)
    .head(10)
)

director_stats_top10

,director,movie_count,inflation_adjusted_gross
8,David Hand,1,5.228953e+09
25,Wolfgang Reitherman,9,3.432920e+09
2,Ben Sharpsteen,1,2.188229e+09
26,full credits,1,2.187091e+09
20,Ron Clements,7,1.318950e+09
12,Hamilton Luske,1,1.236036e+09
23,Wilfred Jackson,2,1.121760e+09
19,Roger Allers,1,7.616409e+08
4,Chris Buck,2,6.988974e+08
10,Gary Trousdale,3,6.791946e+08


### 3.2 Director: Movie Count vs. Box Office Gross

In [35]:
# use top10 dataset to explore movie counts vs inflation adjusted gross from box office

top10 = director_stats_top10.head(10)

base = alt.Chart(top10).encode(
    y=alt.Y('director:N', sort='-x', title='Director'),
    tooltip=['director', 'movie_count', 'inflation_adjusted_gross']
)

movie_chart = base.mark_area(
    opacity=0.2,
    color='steelblue'
).encode(
    x=alt.X('movie_count:Q', title='Number of Movies')
) + base.mark_line(
    color='lightblue',
    point=True
).encode(
    x=alt.X('movie_count:Q')
)

gross_chart = base.mark_area(
    opacity=0.2,
    color='orange'
).encode(
    x=alt.X('inflation_adjusted_gross:Q', title='Total Inflation Adjusted Gross')
) + base.mark_line(
    color='orange',
    point=True
).encode(
    x=alt.X('inflation_adjusted_gross:Q')
)

alt.hconcat(
    movie_chart.properties(title='Movie Count', width=300),
    gross_chart.properties(title='Total Inflation Adjusted Gross', width=400)
).resolve_scale(
    x='independent'
).properties(
    title='Top 10 Directors: Movie Count and Total Inflation Adjusted Gross'
)

alt.HConcatChart(...)


<br>

### Key Findings: Comparison of Directors by Movie Count and Inflation Adjusted Gross

1. I observed that David Hand and Ben Sharpsteen each directed only one movie in the dataset. However, their movies achieved the first and third-highest inflation adjusted gross, respectively. This suggests that commercial success does not necessarily depend on the number of movies a director has made.

2. Gary Trousdale ranked third in terms of movie count but had the lowest total inflation adjusted gross among the directors analyzed. This indicates that directing more movies does not always translate into higher box office revenue.

3. I initially expected Roger Allers, the director of *The Lion King*, to rank among the top-performing directors. However, the analysis shows that he ranks eighth based on inflation adjusted gross

<br>

### 3.3 Top 10 Directors by Average Box Office Gross per Movie

In [36]:
# calculation avergae gross per moive for top 10 directors
director_stats_top10['avg_gross_per_movie'] = (
    director_stats_top10['inflation_adjusted_gross'] / director_stats_top10['movie_count']
)

top10 = director_stats_top10.nlargest(10, 'avg_gross_per_movie')

top10[['director', 'movie_count', 'inflation_adjusted_gross', 'avg_gross_per_movie']]

,director,movie_count,inflation_adjusted_gross,avg_gross_per_movie
8,David Hand,1,5.228953e+09,5.228953e+09
2,Ben Sharpsteen,1,2.188229e+09,2.188229e+09
26,full credits,1,2.187091e+09,2.187091e+09
12,Hamilton Luske,1,1.236036e+09,1.236036e+09
19,Roger Allers,1,7.616409e+08,7.616409e+08
23,Wilfred Jackson,2,1.121760e+09,5.608800e+08
25,Wolfgang Reitherman,9,3.432920e+09,3.814355e+08
4,Chris Buck,2,6.988974e+08,3.494487e+08
10,Gary Trousdale,3,6.791946e+08,2.263982e+08
20,Ron Clements,7,1.318950e+09,1.884214e+08


### 3.4 Top 5 Movies by Average Box Office Gross per Director

In [37]:
# find top 5 from top 10 to have a overall dataset
top5 = (
    top10.head(5)
)

top5_movies = movies_dir_char.merge(
    top5[
        ['director',  'avg_gross_per_movie']
    ],
    on='director',
    how='inner'
)

top5_movies.sort_values(
    by='avg_gross_per_movie',
    ascending=False
)

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross,year,month,day,year_range,director,hero,villian,song,avg_gross_per_movie
0,Snow White and the Seven Dwarfs,1937-12-21,Musical,G,184925485.0,5.228953e+09,1937,December,21,Before 1950,David Hand,Snow White,Evil Queen,Some Day My Prince Will Come,5.228953e+09
1,Pinocchio,1940-02-09,Adventure,G,84300000.0,2.188229e+09,1940,February,9,Before 1950,Ben Sharpsteen,Pinocchio,Stromboli,When You Wish upon a Star,2.188229e+09
2,Fantasia,1940-11-13,Musical,G,83320000.0,2.187091e+09,1940,November,13,Before 1950,full credits,NaN,Chernabog,NaN,2.187091e+09
3,Lady and the Tramp,1955-06-22,Drama,G,93600000.0,1.236036e+09,1955,June,22,1950-1959,Hamilton Luske,Lady and Tramp,Si and Am,Bella Notte,1.236036e+09
4,The Lion King,1994-06-15,Adventure,G,422780140.0,7.616409e+08,1994,June,15,1990-1999,Roger Allers,Simba,Scar,Circle of Life,7.616409e+08


In [38]:
# Sort directors by average gross per movie
top5_ranked = top5_movies.sort_values(
    by='avg_gross_per_movie',
    ascending=False
)

# Create bar chart
chart = alt.Chart(top5_ranked).mark_bar().encode(
   x=alt.X(
    'avg_gross_per_movie:Q',
    title='Average Inflation-Adjusted Gross',
    ),
    y=alt.Y(
        'director:N',
        sort='-x',
        title='Director'
    ),
    tooltip=[
        'director',
        'movie_title',
    ]
).properties(
    title='Top 5 Disney Directors by Average Gross per Movie',
    width=300,
    height=250
)

chart

alt.Chart(...)


<br>

# Conclusion

###
Before conducting the analysis, we expected action and adventure movies to be the best-selling genres. The genre analysis shows that our initial assumption was correct. We also initially predicted that *The Lion King* would be one of Disney's best-performing movies and that its director, Roger Allers, would rank among the top directors.

However, when examining movie performance based on inflation-adjusted gross, we discovered some interesting results. The three highest-grossing Disney movies are *Snow White and the Seven Dwarfs*, *Pinocchio*, and *Fantasia*. All three movies were released before 1950.

The Lion King ranks fifth, following Lady and the Tramp, among the top five Disney movies by inflation adjusted gross.

This suggests that Disney's early animated films achieved extraordinary long-term box office success, demonstrating their lasting popularity and commercial impact across generations.

# References

## Resources used
* [Data Visualization](https://www.kaggle.com/code/mrigendraagrawal/eda-and-data-storytelling-on-disney-movies/notebook)
    * Using genre as a category to present the movies popularity over generations by **shiroe**.
* [Data Visualization](https://www.kaggle.com/code/edumisvieramartin/end-to-end-project-disney-movies-classification/notebook)
    * Using the inflation adjusted gross for analyzing the best performance of directors and movies taken from **Edumis Viera**.

![](img/disney_movies.jpg)

Image obtained from [here](https://github.com/sibeltan/disney_movies_analysis/blob/master/disney_movies.jpg)